# YOLO11s-seg Rectal Tumor Training — Kaggle Edition

**Before running:**
1. Upload `rectal_tumor_dataset.zip` as a Kaggle Dataset → get its name (e.g. `username/rectal-tumor-dataset`)
2. Upload `rectal_tumor_code.zip` as a Kaggle Dataset → get its name (e.g. `username/rectal-tumor-code`)
3. Attach both datasets to this notebook (Add Input → search by name)
4. Replace `{KAGGLE_DATASET_NAME}` and `{KAGGLE_CODE_NAME}` below
5. Choose GPU T4 x2 as accelerator (Settings → Accelerator)
6. Run All or step through cells

## Cell 1: Install Dependencies

In [ ]:
!pip install -q ultralytics opencv-python numpy tqdm pyyaml Pillow thop rich pandas matplotlib

## Cell 2: Configuration — Replace Placeholders

In [ ]:
import os
import sys
import shutil
import yaml
from pathlib import Path

# ============================================================
# REPLACE THESE with your actual Kaggle Dataset names
# ============================================================
KAGGLE_DATASET_NAME = "{KAGGLE_DATASET_NAME}"   # e.g. "johndoe/rectal-tumor-dataset"
KAGGLE_CODE_NAME    = "{KAGGLE_CODE_NAME}"      # e.g. "johndoe/rectal-tumor-code"
# ============================================================

KAGGLE_INPUT  = Path("/kaggle/input")
KAGGLE_WORKING = Path("/kaggle/working")

DATASET_PATH = KAGGLE_INPUT / KAGGLE_DATASET_NAME.split("/")[-1]
CODE_PATH    = KAGGLE_INPUT / KAGGLE_CODE_NAME.split("/")[-1]

print(f"Dataset path: {DATASET_PATH}  (exists: {DATASET_PATH.exists()})")
print(f"Code path:    {CODE_PATH}  (exists: {CODE_PATH.exists()})")

# Copy code to working dir (Kaggle input is read-only)
WORKING_CODE = KAGGLE_WORKING / "rectal-tumor-system"
if not WORKING_CODE.exists():
    shutil.copytree(CODE_PATH, WORKING_CODE)
    print(f"Code copied to {WORKING_CODE}")
else:
    print(f"Code already at {WORKING_CODE}")

os.chdir(WORKING_CODE)
sys.path.insert(0, str(WORKING_CODE))
print(f"CWD: {os.getcwd()}")

## Cell 3: Patch Dataset YAML for Kaggle Paths

In [ ]:
# Create Kaggle-compatible dataset YAML
kaggle_yaml = WORKING_CODE / "configs" / "rectal_tumor_kaggle.yaml"

config = {
    "path": str(DATASET_PATH.resolve()),
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "names": {0: "tumor"},
    "nc": 1,
}

with open(kaggle_yaml, "w") as f:
    yaml.safe_dump(config, f)

print(f"Patched YAML written to {kaggle_yaml}")
print(open(kaggle_yaml).read())

# Verify dataset structure
for split in ["images/train", "images/val", "images/test"]:
    p = DATASET_PATH / split
    if p.exists():
        count = len(list(p.rglob("*")))
        print(f"  {split}: {count} files")
    else:
        print(f"  WARNING: {split} not found!")

## Cell 4: Train YOLO11s-seg

In [ ]:
# Train with GPU (T4 x2). Expected duration: 3-6 hours for 200 epochs.
# Adjust --epochs downward for a quick test (e.g. 20).

!python train/train_baseline.py \
    --model yolo11s-seg.pt \
    --data configs/rectal_tumor_kaggle.yaml \
    --epochs 200 \
    --imgsz 640 \
    --batch 16 \
    --device 0 \
    --project runs/baseline \
    --name yolo11s_seg_v1 \
    --no-post

## Cell 5: Per-Subset Evaluation

In [ ]:
BEST_PT = "runs/baseline/yolo11s_seg_v1/weights/best.pt"

!python eval/eval_per_subset.py \
    --weights {BEST_PT} \
    --data-root {DATASET_PATH} \
    --device 0 \
    --output-dir results

## Cell 6: Visualize Predictions (per subset, 8 samples each)

In [ ]:
!python eval/visualize_predictions.py \
    --weights {BEST_PT} \
    --data-root {DATASET_PATH} \
    --num-samples 8 \
    --conf 0.25 \
    --output-dir results/visualizations

## Cell 7: Speed Benchmark (GPU + CPU)

In [ ]:
!python eval/speed_benchmark.py \
    --weights {BEST_PT} \
    --warmup 10 \
    --runs 100 \
    --output results/speed_benchmark.md

## Cell 8: Package Results for Download

In [ ]:
import zipfile

OUTPUT_ZIP = KAGGLE_WORKING / "rectal_tumor_results.zip"

with zipfile.ZipFile(OUTPUT_ZIP, "w", zipfile.ZIP_DEFLATED) as zf:
    # best.pt
    best = Path(BEST_PT)
    if best.exists():
        zf.write(best, best.name)
        print(f"+ {best.name}  ({best.stat().st_size / 1e6:.1f} MB)")
    
    # Metrics
    for f in ["results/per_subset_metrics.md", "results/per_subset_metrics.csv"]:
        p = Path(f)
        if p.exists():
            zf.write(p, str(p))
            print(f"+ {p}")
    
    # Visualizations
    vis_dir = Path("results/visualizations")
    if vis_dir.exists():
        for img in vis_dir.rglob("*"):
            if img.is_file():
                zf.write(img, str(img))
        print(f"+ {len(list(vis_dir.rglob('*')))} visualization files")
    
    # Speed benchmark
    sb = Path("results/speed_benchmark.md")
    if sb.exists():
        zf.write(sb, str(sb))
        print(f"+ {sb}")
    
    # Training log
    for log in Path("logs").glob("*.log"):
        zf.write(log, str(log))
        print(f"+ {log}")

size_mb = OUTPUT_ZIP.stat().st_size / 1e6
print(f"\nResults packaged: {OUTPUT_ZIP} ({size_mb:.1f} MB)")
print("Download this file from the Kaggle Output tab (Data → Output → Download)")